# AdversarialSearchMCCFRV31Agent

This notebook explains the major components of `AdversarialSearchMCCFRV31Agent`, how the saved policy is supposed to be trained, and how data moves through the agent across **preflop**, **flop**, **turn**, and **river**.

Two important facts from the repository:

- The deployed `v3.1` agent is an **inference-time policy player** whose decision logic lives almost entirely inside `declare_action()`.
- The saved policy file still contains richer MCCFR-style training artifacts (`regret_sum`, `strategy_sum`, `action_weights`, refinement metadata), but the current `v3.1` runtime agent only reads `action_weights` and ignores the rest.
- The current trainer script still tries to construct `AdversarialSearchMCCFRV31Agent(..., training_enabled=True)`, but the runtime class in this folder does **not** accept `training_enabled`. So the repo contains a training/runtime mismatch.


## 1. Major Components

At runtime, `AdversarialSearchMCCFRV31Agent` is best understood as five components:

1. **Policy loader**
   - Loads `adversarial_search_mccfr_v3_1_policy.json` into `self.policy_data`.
2. **State summarizer**
   - Extracts stack, pot, amount-to-call, position, active player count, legal actions, and opponent action statistics from `round_state`.
3. **Equity estimator**
   - Uses a preflop heuristic before the board appears.
   - Uses Monte Carlo equity estimates on flop/turn/river in the original code.
4. **Linear policy scorer**
   - Builds a feature vector and scores `fold`, `call`, and `raise` with learned action weights.
5. **Safety regularizer**
   - Computes a conservative anchor action.
   - Blends the learned distribution back toward the anchor, especially when opponent evidence is weak or the spot is expensive.

So the runtime shape is:

`round_state -> features -> weighted action scores -> softmax -> anchor regularization -> final legal action`


In [ ]:
from pathlib import Path
import inspect
import json

base = Path.cwd()
policy_path = base / 'adversarial_search_mccfr_v3_1_policy.json'
agent_path = base / 'adversarial_search_mccfr_v3_1_agent.py'
train_path = base / 'train_adversarial_search_mccfr_v3_1_agent.py'

with policy_path.open('r', encoding='utf-8') as f:
    policy = json.load(f)

summary = {
    'policy_top_level_keys': sorted(policy.keys()),
    'action_weight_actions': sorted(policy['action_weights'].keys()),
    'regret_state_count': len(policy.get('regret_sum', {})),
    'strategy_state_count': len(policy.get('strategy_sum', {})),
}
summary

## 2. How It Is Supposed To Be Trained

The **design lineage** in this repo is:

- `ConditionThresholdPlayer`: rule-based anchor.
- `AdvancedCFRPlayer`: adds regret tables and average strategy over abstracted information sets.
- `DiscountedMCCFRPlusAgent`: discounted regrets and stronger priors.
- `LearnableDiscountedMCCFRAgent`: learns linear action weights online from round outcomes.
- `AdversarialSearchMCCFRV2Agent` / `V3Agent`: add search/refinement layers and adaptive thresholds.
- `AdversarialSearchMCCFRV31Agent` in this folder: simplified, tournament-safe runtime policy.

The saved `v3.1` policy file still looks like it came from the richer online-training family. The intended training loop is:

1. Play many heads-up games against a mixed opponent pool.
2. For each decision point, build an information set key and a feature vector.
3. Sample an action from the current strategy during training.
4. At round end, compute reward from chip delta.
5. Update regrets, average strategy sums, and linear action weights.
6. Periodically discount old values and save the learned policy.

The trainer script in this folder rotates through this mixed pool:

- `condition_threshold_player`
- `advanced_cfr_player`
- `learnable_discounted_mccfr_agent`
- `adversarial_search_mccfr_v2_agent`

But in the **current checked-in code**, the runtime `AdversarialSearchMCCFRV31Agent` constructor does not expose training hooks, so the trainer no longer matches the class interface.

In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location('agent_mod', agent_path)
agent_mod = importlib.util.module_from_spec(spec)
sys.modules['agent_mod'] = agent_mod
spec.loader.exec_module(agent_mod)

signature = inspect.signature(agent_mod.AdversarialSearchMCCFRV31Agent.__init__)
print(signature)
print('training_enabled' in signature.parameters)

## 3. Data Flow By Poker Street

The data flow is the same shape on every street, but the **equity source** changes:

- **Preflop**
  - Input: hole cards, action history, pot, stack, position.
  - Equity proxy: handcrafted preflop strength heuristic.
  - Typical effect: more weight on pair/high-card/suited/connector structure.
- **Flop**
  - Input adds 3 community cards.
  - Equity proxy: rollout estimate with board context.
  - Typical effect: draw potential, pressure, and pot odds become more important.
- **Turn**
  - Input adds the 4th community card.
  - Equity estimate gets sharper.
  - Typical effect: stronger separation between value hands and marginal bluff-catchers.
- **River**
  - Input adds the 5th community card.
  - Equity estimate is most final.
  - Typical effect: conservative pressure adjustment; thin raises are penalized more heavily.

Per decision, the runtime `v3.1` code effectively does:

1. Read `round_state`.
2. Derive `to_call`, `pot_size`, `stack`, `position`, `opponent_stats`, `reliability`.
3. Estimate `equity`.
4. Build features:
   - `equity`, `equity_centered`, `pot_odds`, `stack_pressure`, `position`, `street`, `aggression`, `fold_rate`, `reliability`, `raise_ratio`, `free_call`
5. Score each legal action with linear weights.
6. Convert scores to probabilities with softmax.
7. Blend toward a safe anchor policy.
8. Apply an extra anti-spew rule against weak raises.
9. Pick the highest-probability legal action.


In [ ]:
import math
from pprint import pprint

RANK_VALUE = {
    '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9,
    'T': 10, 'J': 11, 'Q': 12, 'K': 13, 'A': 14,
}

DEFAULT_WEIGHTS = {
    'fold': {
        'bias': 0.10, 'equity': -2.40, 'equity_centered': -1.20, 'pot_odds': -0.50,
        'stack_pressure': 1.40, 'position': -0.05, 'street': 0.05, 'aggression': -0.10,
        'fold_rate': -0.05, 'reliability': -0.05, 'raise_ratio': 0.70, 'free_call': -2.00,
    },
    'call': {
        'bias': 0.08, 'equity': 1.60, 'equity_centered': 0.90, 'pot_odds': -0.85,
        'stack_pressure': -0.30, 'position': 0.10, 'street': 0.05, 'aggression': 0.18,
        'fold_rate': -0.08, 'reliability': 0.08, 'raise_ratio': -0.12, 'free_call': 0.80,
    },
    'raise': {
        'bias': -0.02, 'equity': 2.15, 'equity_centered': 1.25, 'pot_odds': -0.50,
        'stack_pressure': -0.55, 'position': 0.18, 'street': 0.12, 'aggression': -0.05,
        'fold_rate': 0.30, 'reliability': 0.12, 'raise_ratio': -0.30, 'free_call': 0.10,
    },
}

for action, loaded in policy.get('action_weights', {}).items():
    if action in DEFAULT_WEIGHTS:
        DEFAULT_WEIGHTS[action].update({k: float(v) for k, v in loaded.items() if k in DEFAULT_WEIGHTS[action]})

def clamp(value, lo, hi):
    return max(lo, min(hi, value))

def rank_value(rank_char):
    return RANK_VALUE[rank_char]

def preflop_strength(hole_card, in_position=False):
    ranks = sorted([rank_value(card[1]) for card in hole_card], reverse=True)
    high, low = ranks
    suited = hole_card[0][0] == hole_card[1][0]
    gap = high - low
    score = 0.35
    if high == low:
        score += 0.25 + min(high, 12) * 0.028
    else:
        score += max(high - 8, 0) * 0.045
        score += max(low - 8, 0) * 0.03
    if suited:
        score += 0.05
    if gap == 1:
        score += 0.05
    elif gap == 2:
        score += 0.025
    elif gap >= 4:
        score -= 0.05
    if high >= 12 and low >= 10:
        score += 0.06
    if high == 14 and low >= 10:
        score += 0.04
    if high < 10 and low < 8 and not suited:
        score -= 0.07
    if in_position:
        score += 0.02
    return clamp(score, 0.0, 0.95)

def street_equity_proxy(street, hole_card, community_card, in_position=False):
    # Notebook-friendly fake equity model.
    if street == 'preflop':
        return preflop_strength(hole_card, in_position)
    base = preflop_strength(hole_card, in_position)
    board_bonus = min(0.18, 0.03 * len(community_card))
    broadway_bonus = 0.04 * sum(card[1] in {'T', 'J', 'Q', 'K', 'A'} for card in hole_card)
    return clamp(base + board_bonus + broadway_bonus - (0.02 if street == 'river' else 0.0), 0.0, 0.98)

def pot_size(round_state):
    return round_state['pot']['main']['amount'] + sum(side['amount'] for side in round_state['pot']['side'])

def amount_to_call(round_state, my_uuid='hero'):
    histories = round_state['action_histories'].get(round_state['street'], [])
    highest_amount = 0
    my_amount = 0
    for action in histories:
        if action is None or 'amount' not in action:
            continue
        amount = action['amount']
        highest_amount = max(highest_amount, amount)
        if action.get('uuid') == my_uuid:
            my_amount = amount
    return max(0, highest_amount - my_amount)

def has_position(round_state, my_uuid='hero'):
    seats = round_state['seats']
    my_index = next((i for i, seat in enumerate(seats) if seat['uuid'] == my_uuid), None)
    if my_index is None:
        return False
    dealer_btn = round_state['dealer_btn']
    if len(seats) == 2:
        return my_index == dealer_btn and round_state['street'] != 'preflop'
    return my_index == (dealer_btn - 1) % len(seats)

def my_stack(round_state, my_uuid='hero'):
    return next(seat['stack'] for seat in round_state['seats'] if seat['uuid'] == my_uuid)

def legal_actions(valid_actions):
    return {entry['action'] for entry in valid_actions}

def raise_amount_bounds(valid_actions):
    for entry in valid_actions:
        if entry['action'] == 'raise' and isinstance(entry.get('amount'), dict):
            return entry['amount'].get('min', 0), entry['amount'].get('max', 0)
    return 0, 0

def opponent_stats(round_state, my_uuid='hero'):
    stats = {'raise': 0, 'call': 0, 'fold': 0, 'total': 0}
    for street_actions in round_state['action_histories'].values():
        for action in street_actions:
            if not action or action.get('uuid') == my_uuid:
                continue
            move = action.get('action')
            if move in ('raise', 'call', 'fold'):
                stats[move] += 1
                stats['total'] += 1
    return stats

def opponent_reliability(stats):
    total = stats['total']
    return clamp(0.20 + total / (total + 18.0), 0.20, 1.0)

def board_pressure_adjustment(street, to_call_value, stack_value):
    adjustment = 0.0
    if street == 'river':
        adjustment -= 0.01
    if stack_value > 0 and to_call_value / max(stack_value, 1) >= 0.30:
        adjustment -= 0.04
    return adjustment

def feature_map(round_state, valid_actions, hole_card):
    street = round_state['street']
    to_call_value = amount_to_call(round_state)
    pot_value = pot_size(round_state)
    stack_value = my_stack(round_state)
    stats = opponent_stats(round_state)
    reliability = opponent_reliability(stats)
    in_position = has_position(round_state)
    equity = street_equity_proxy(street, hole_card, round_state['community_card'], in_position)
    equity = clamp(equity + 0.015 * float(in_position) + board_pressure_adjustment(street, to_call_value, stack_value), 0.0, 1.0)
    pot_odds = to_call_value / max(pot_value + to_call_value, 1)
    stack_pressure = to_call_value / max(stack_value, 1)
    aggression = stats['raise'] / max(stats['total'], 1)
    fold_rate = stats['fold'] / max(stats['total'], 1)
    street_index = {'preflop': 0.0, 'flop': 1.0, 'turn': 2.0, 'river': 3.0}[street]
    min_raise, _ = raise_amount_bounds(valid_actions)
    raise_ratio = min_raise / max(pot_value + to_call_value, 1)
    return {
        'bias': 1.0,
        'equity': equity,
        'equity_centered': equity - 0.5,
        'pot_odds': pot_odds,
        'stack_pressure': stack_pressure,
        'position': 1.0 if in_position else 0.0,
        'street': street_index / 3.0,
        'aggression': aggression,
        'fold_rate': fold_rate,
        'reliability': reliability,
        'raise_ratio': raise_ratio,
        'free_call': 1.0 if to_call_value == 0 else 0.0,
    }

def weighted_score(action_name, features):
    return sum(DEFAULT_WEIGHTS[action_name].get(k, 0.0) * v for k, v in features.items())

def softmax(score_map):
    m = max(score_map.values())
    exps = {k: math.exp(v - m) for k, v in score_map.items()}
    total = sum(exps.values())
    return {k: v / total for k, v in exps.items()}

def safe_anchor_action(round_state, valid_actions, hole_card):
    street = round_state['street']
    to_call_value = amount_to_call(round_state)
    pot_value = pot_size(round_state)
    stack_value = my_stack(round_state)
    stats = opponent_stats(round_state)
    equity = feature_map(round_state, valid_actions, hole_card)['equity']
    aggression = stats['raise'] / max(stats['total'], 1)
    raise_threshold = 0.74
    call_threshold = 0.51
    if aggression >= 0.35:
        raise_threshold += 0.03
        call_threshold -= 0.03
    elif aggression <= 0.15:
        raise_threshold -= 0.02
        call_threshold += 0.01
    if to_call_value > 0:
        pot_odds = to_call_value / max(pot_value + to_call_value, 1)
        stack_ratio = to_call_value / max(stack_value, 1)
        call_threshold = max(call_threshold, pot_odds + 0.06)
        raise_threshold = max(raise_threshold, pot_odds + 0.18)
        if stack_ratio >= 0.35:
            call_threshold += 0.07
            raise_threshold += 0.05
        elif stack_ratio <= 0.08:
            call_threshold -= 0.02
    legal = legal_actions(valid_actions)
    if to_call_value == 0:
        if 'raise' in legal and equity >= raise_threshold - 0.06:
            return 'raise'
        return 'call'
    if 'raise' in legal and equity >= raise_threshold:
        return 'raise'
    if equity >= call_threshold:
        return 'call'
    return 'fold'

def runtime_v31_decision(round_state, valid_actions, hole_card):
    features = feature_map(round_state, valid_actions, hole_card)
    legal = legal_actions(valid_actions)
    candidate_scores = {action: weighted_score(action, features) for action in sorted(legal)}
    learned_distribution = softmax(candidate_scores)
    anchor_action = safe_anchor_action(round_state, valid_actions, hole_card)
    reliability = features['reliability']
    anchor_mass = clamp(0.58 - 0.18 * reliability, 0.32, 0.62)
    regularized = {}
    for action in legal:
        regularized[action] = (1.0 - anchor_mass) * learned_distribution.get(action, 0.0) + anchor_mass * (1.0 if action == anchor_action else 0.0)
    if 'raise' in regularized and anchor_action != 'raise':
        raise_support = regularized.get('raise', 0.0)
        call_support = regularized.get('call', 0.0)
        to_call_value = amount_to_call(round_state)
        stack_value = my_stack(round_state)
        thin_raise = (
            features['equity'] < 0.60
            or reliability < 0.45
            or to_call_value / max(stack_value, 1) > 0.25
        )
        if thin_raise and raise_support < call_support + 0.10:
            regularized['raise'] *= 0.60
            if 'call' in regularized:
                regularized['call'] += raise_support * 0.40
    total = sum(regularized.values())
    regularized = {k: v / total for k, v in regularized.items()}
    chosen = max(regularized, key=regularized.get)
    return {
        'features': features,
        'candidate_scores': candidate_scores,
        'learned_distribution': learned_distribution,
        'anchor_action': anchor_action,
        'regularized_distribution': regularized,
        'chosen_action': chosen,
    }


## 4. Fake Round States For Preflop / Flop / Turn / River

These examples use fake states but keep the same data shape the real agent consumes from PyPokerEngine.

In [ ]:
VALID_ACTIONS = [
    {'action': 'fold', 'amount': 0},
    {'action': 'call', 'amount': 0},
    {'action': 'raise', 'amount': {'min': 20, 'max': 200}},
]

def make_round_state(street, community_card, hero_stack, villain_stack, main_pot, histories, dealer_btn=1):
    return {
        'street': street,
        'community_card': community_card,
        'dealer_btn': dealer_btn,
        'pot': {'main': {'amount': main_pot}, 'side': []},
        'action_histories': histories,
        'seats': [
            {'uuid': 'hero', 'stack': hero_stack, 'state': 'participating'},
            {'uuid': 'villain', 'stack': villain_stack, 'state': 'participating'},
        ],
    }

examples = {
    'preflop': {
        'hole_card': ['SA', 'SK'],
        'round_state': make_round_state(
            'preflop',
            [],
            hero_stack=190,
            villain_stack=200,
            main_pot=15,
            histories={'preflop': [{'uuid': 'villain', 'action': 'raise', 'amount': 10}]},
        ),
    },
    'flop': {
        'hole_card': ['HA', 'HQ'],
        'round_state': make_round_state(
            'flop',
            ['H2', 'D9', 'CT'],
            hero_stack=160,
            villain_stack=160,
            main_pot=40,
            histories={
                'preflop': [{'uuid': 'villain', 'action': 'call', 'amount': 10}],
                'flop': [{'uuid': 'villain', 'action': 'raise', 'amount': 20}],
            },
        ),
    },
    'turn': {
        'hole_card': ['S9', 'S8'],
        'round_state': make_round_state(
            'turn',
            ['S6', 'D7', 'CK', 'S2'],
            hero_stack=120,
            villain_stack=140,
            main_pot=70,
            histories={
                'preflop': [{'uuid': 'villain', 'action': 'call', 'amount': 10}],
                'flop': [{'uuid': 'villain', 'action': 'call', 'amount': 20}],
                'turn': [{'uuid': 'villain', 'action': 'raise', 'amount': 30}],
            },
        ),
    },
    'river': {
        'hole_card': ['C4', 'D4'],
        'round_state': make_round_state(
            'river',
            ['H4', 'S9', 'CQ', 'D2', 'C7'],
            hero_stack=90,
            villain_stack=110,
            main_pot=100,
            histories={
                'preflop': [{'uuid': 'villain', 'action': 'call', 'amount': 10}],
                'flop': [{'uuid': 'villain', 'action': 'call', 'amount': 20}],
                'turn': [{'uuid': 'villain', 'action': 'call', 'amount': 30}],
                'river': [{'uuid': 'villain', 'action': 'raise', 'amount': 40}],
            },
        ),
    },
}

for street, payload in examples.items():
    result = runtime_v31_decision(payload['round_state'], VALID_ACTIONS, payload['hole_card'])
    print(f'--- {street.upper()} ---')
    print('chosen_action:', result['chosen_action'])
    print('anchor_action:', result['anchor_action'])
    print('features:')
    pprint({k: round(v, 4) for k, v in result['features'].items()})
    print('regularized_distribution:', {k: round(v, 4) for k, v in result['regularized_distribution'].items()})
    print()


## 5. What Changes Across Streets

The examples above should show a few street-specific patterns:

- **Preflop** depends mainly on hole-card structure and price.
- **Flop** begins to react to board interaction and recent aggression.
- **Turn** usually increases commitment pressure because the pot is larger and stacks are shallower.
- **River** keeps the same feature shape, but the runtime logic adds extra conservatism in expensive spots and weak-raise situations.


## 6. Toy Training Update

The current runtime class does not train online, but the saved policy structure comes from a richer MCCFR-style family. The next cell shows the **kind** of update that produces those tables:

- maintain regrets per abstract state
- compute counterfactual utilities
- update regrets using realized reward
- adjust action weights toward better actions

This is deliberately small and fake, but it demonstrates the core idea.

In [ ]:
toy_regret_sum = {'state_1': {'fold': 0.0, 'call': 0.0, 'raise': 0.0}}
toy_strategy = {'fold': 0.15, 'call': 0.45, 'raise': 0.40}
toy_utilities = {'fold': -0.20, 'call': 0.35, 'raise': 0.60}
chosen_action = 'raise'
chosen_prob = 0.40
reward = 0.75

importance = min(2.5, 1.0 / chosen_prob)
adjusted_utilities = dict(toy_utilities)
adjusted_utilities[chosen_action] = 0.7 * reward * importance + 0.3 * adjusted_utilities[chosen_action]
node_value = sum(toy_strategy[a] * adjusted_utilities[a] for a in toy_strategy)

for action in toy_strategy:
    toy_regret_sum['state_1'][action] = max(0.0, toy_regret_sum['state_1'][action] + adjusted_utilities[action] - node_value)

print('adjusted_utilities:', adjusted_utilities)
print('node_value:', round(node_value, 4))
print('updated_regrets:', toy_regret_sum['state_1'])


In [ ]:
toy_features = {
    'bias': 1.0,
    'equity': 0.68,
    'equity_centered': 0.18,
    'pot_odds': 0.20,
    'stack_pressure': 0.10,
    'position': 1.0,
    'street': 2.0 / 3.0,
    'aggression': 0.30,
    'fold_rate': 0.20,
    'reliability': 0.55,
    'raise_ratio': 0.25,
    'free_call': 0.0,
}

toy_weights = {action: dict(weights) for action, weights in DEFAULT_WEIGHTS.items()}
targets = {'fold': -0.25, 'call': 0.20, 'raise': 0.65}
learning_rate = 0.05

def score(weights, features):
    return sum(weights[k] * features[k] for k in features)

before = {a: score(toy_weights[a], toy_features) for a in toy_weights}

for action in toy_weights:
    prediction = score(toy_weights[action], toy_features)
    error = max(-2.0, min(2.0, targets[action] - prediction))
    step_scale = 1.0 if action == 'raise' else 0.30
    for feature_name in toy_features:
        toy_weights[action][feature_name] += learning_rate * step_scale * error * toy_features[feature_name]

after = {a: score(toy_weights[a], toy_features) for a in toy_weights}
print('scores_before:', {k: round(v, 4) for k, v in before.items()})
print('scores_after :', {k: round(v, 4) for k, v in after.items()})


## 7. Concise Takeaways

- **Major runtime logic**: feature extraction, equity estimate, linear action scoring, anchor regularization, legal-action selection.
- **Training intent**: MCCFR-style regret updates plus learned action-weight updates from realized round reward.
- **Per-street flow**: same pipeline every street, but preflop uses a heuristic strength model while later streets rely on more board-aware equity.
- **Important repo reality**: the checked-in `v3.1` runtime player is simpler than the saved policy schema and simpler than the trainer script expects.
